# TTBB Training on Colab — Woodruff North

Fine-tunes a PPO agent (warm-started from vanilla) against the frozen CEIA baseline as team-1 opponent. Long-match episodes (2000 steps) with cumulative goal-differential reward.

**Runtime:** A100 (Runtime → Change runtime type → GPU → A100)

**Run order:**
1. Cells 1–2 (GPU check + conda install). Cell 2 triggers a kernel restart — after the restart banner, continue from cell 3.
2. Cells 3–8 install the stack, clone the repo, mount Drive, smoke-test, and configure training. Run them in order.
3. Cell 9 is the actual training (6h). Cell 10 extracts the final checkpoint after training finishes.

**Entropy choice:** `entropy_coeff = 0.001` (Option B — low exploration pressure to preserve vanilla's competent argmax actions).

## 1. GPU check

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install conda (triggers kernel restart)

After the kernel restart banner, **continue from cell 3** — don't re-run this cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Headless display (Xvfb)

In [ ]:
import subprocess, shutil, os
if shutil.which('Xvfb') is None:
    !apt-get install -y --no-install-recommends xvfb > /dev/null 2>&1
if shutil.which('Xvfb') is None:
    raise SystemExit('Xvfb still not found')
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x720x24'])
os.environ['DISPLAY'] = ':99'
print('xvfb ready on :99')

## 4. Python 3.8 env + all dependencies

One consolidated cell — order matters: pandas/numpy/gym first, then soccer-twos (which brings ray+mlagents), then force-reinstall numpy (so `np.bool` still works) and force-reinstall torch (so A100 kernels work). Takes ~4 min.

In [ ]:
!conda create -n ttbb python=3.8 -y -q 2>&1 | tail -3
%env PATH=/usr/local/envs/ttbb/bin:/usr/local/bin:/usr/bin:/bin
!python --version

In [ ]:
# 1. Pinned build tools (per starter README)
!pip install --quiet pip==23.3.2 setuptools==65.5.0 wheel==0.38.4
!pip cache purge > /dev/null

# 2. Ray-Tune + RLlib soft deps + gym pinned to what soccer-twos expects
!pip install --quiet \
    pandas tabulate dm-tree scipy lz4 opencv-python-headless tensorboardX \
    'gym==0.19.0'

# 3. Soccer-twos env + compatible ray/mlagents stack (PACE-matching pins)
!pip install --quiet \
    soccer-twos==0.1.14 \
    ray==1.4.0 \
    mlagents-envs==0.27.0 \
    protobuf==3.20.3 \
    pydantic==1.10.13 \
    prometheus-client==0.11.0

# 4. Force numpy to a version that still exposes np.bool (mlagents uses it)
!pip install --quiet --force-reinstall --no-deps numpy==1.23.5

# 5. Force-install torch with CUDA 11.3 kernels for A100 (soccer-twos pulls torch 1.8.1+cu102 which does NOT work on sm_80)
!pip install --quiet --force-reinstall --no-deps \
    --extra-index-url https://download.pytorch.org/whl/cu113 \
    torch==1.11.0+cu113 torchvision==0.12.0+cu113

# 6. Sanity checks
!python -c "import numpy as np; print('numpy', np.__version__, 'np.bool:', np.bool)"
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); x = torch.randn(64,64,device='cuda'); _ = x @ x.T; print('A100 matmul OK')"
!python -c "import soccer_twos, ray; print('soccer_twos OK, ray', ray.__version__)"

## 5. Clone the project repo

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Luca65537/soccer-twos-woodruffnorth.git'
!rm -rf /content/repo
!git clone -q $GITHUB_REPO_URL /content/repo
%cd /content/repo/training
!ls -la | head -20

## 6. Mount Drive for checkpoint persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/soccer_twos/ray_results
# Symlink so Ray checkpoints land on Drive
!ln -sfn /content/drive/MyDrive/soccer_twos/ray_results /content/repo/training/ray_results
!ls -la /content/repo/training/ray_results

## 7. Smoke test — single long match vs baseline

~2 min. Confirms Unity launches headless, baseline wrapper runs, reward flows. Expect `-10` to `-20` total reward with random team-0 actions.

In [ ]:
!xvfb-run -a -s '-screen 0 1280x720x24' python -u smoke_ttbb.py

## 8. Configure training for A100 + Option B (low entropy)

- 4 rollout workers × 2 envs/worker = 8 parallel Unity envs
- `entropy_coeff = 0.001` (10× lower than PACE default — preserves vanilla's confident argmax actions)
- Budget: 20M env steps, max 6h wall-clock
- Checkpoint every 10 iterations to Drive

In [ ]:
import re, pathlib
p = pathlib.Path('train_vs_baseline.py')
s = p.read_text()
s = re.sub(r'NUM_ENVS_PER_WORKER = \d+', 'NUM_ENVS_PER_WORKER = 2', s)
s = re.sub(r'"num_workers": \d+', '"num_workers": 4', s)
s = re.sub(r'"entropy_coeff": [0-9.e\-]+', '"entropy_coeff": 0.001', s)
s = re.sub(r'"timesteps_total": \d+', '"timesteps_total": 20000000', s)
s = re.sub(r'"time_total_s": \d+', '"time_total_s": 21600', s)  # 6h
s = re.sub(r'checkpoint_freq=\d+', 'checkpoint_freq=10', s)
p.write_text(s)
!grep -E 'num_workers|NUM_ENVS_PER_WORKER|entropy_coeff|timesteps_total|time_total_s|checkpoint_freq' train_vs_baseline.py

## 9. Launch training (6h)

Log is tee'd to Drive so you can inspect after session timeout. Within ~1 min of launch you should see `Warm-start: missing=[], unexpected=[]` and then `Warm-start complete: vanilla -> trainable policy synced`. First iteration result appears around 25 s later.

In [ ]:
!xvfb-run -a -s '-screen 0 1280x720x24' python -u train_vs_baseline.py 2>&1 | tee /content/drive/MyDrive/soccer_twos/train_log_$(date +%Y%m%d_%H%M).txt

## 10. Extract the best checkpoint into the submission format

After training, this saves `ttbb_checkpoint.pth` (matching `agent_vanilla/model.py`) to Drive. Drop it into a fresh `WOODRUFF_NORTH_AGENT/` folder as `checkpoint.pth` and re-zip to replace the Gradescope upload.

In [ ]:
import glob, os
# Find best checkpoint by (implicit) iteration number — last one
ckpts = sorted(glob.glob('ray_results/ttbb_long/*/checkpoint_*/checkpoint-*'))
ckpts = [c for c in ckpts if not c.endswith('.tune_metadata')]
assert ckpts, 'No checkpoints found under ray_results/ttbb_long/'
latest = ckpts[-1]
print('Latest checkpoint:', latest)
!python extract_ttbb_to_standalone.py '$latest' /content/drive/MyDrive/soccer_twos/ttbb_checkpoint.pth
!ls -la /content/drive/MyDrive/soccer_twos/ttbb_checkpoint.pth